# Ensemble Methods for Classification

In [ ]:
%%bash

pip install --upgrade 'scikit-learn<1.6.0'

## Exercise 1: Load the data

Use Pandas to load the training and testing data.

In [ ]:
%%bash

ls /kaggle/input/digit-recognizer

### Solution

In [ ]:
import pandas as pd


TRAIN_DF = pd.read_csv(
    "/kaggle/input/digit-recognizer/train.csv"
)

TEST_FEATURES_DF = pd.read_csv(
    "/kaggle/input/digit-recognizer/test.csv"
)


## Exercise 2: Train/Val Split

Split the data into a training set and a validation set. The size of the validation set should be half the size of the test set. Make sure to use stratified sampling so that the distribution of classes is the same in both the training and validation sets.

### Solution

In [ ]:
import numpy as np
from sklearn import model_selection


PRNG = np.random.RandomState(42)
n_test_samples, _ = TEST_FEATURES_DF.shape
n_val_samples = n_test_samples // 2
_stratify = TRAIN_DF.loc[:, "label"]

train_df, val_df = model_selection.train_test_split(
    TRAIN_DF,
    random_state=PRNG,
    test_size=n_val_samples,
    stratify=_stratify,
)

train_features_df = train_df.drop("label", axis=1)
train_label = train_df.loc[:, "label"]

val_features_df = val_df.drop("label", axis=1)
val_label = val_df.loc[:, "label"]


## Exercise 3: Prepare the data for ML

Create a data preparation pipeline to prepare the raw training data for machine learning algorithms. Your data preparation pipeline should have three stages.

1. **Data Cleaning:** use [impute.SimpleImputer](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html) to make sure your pipeline handles missing values properly.
2. **Feature Selection:** use [feature_selection.VarianceThreshold](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.VarianceThreshold.html) to make sure that you pipeline removes any useless features from the training data.
3. **Feature Scaling:** use [preprocessing.StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) to make sure that the features have mean zero.

Your data preparation pipeline should return a Pandas DataFrame.

### Solution

In [ ]:
from sklearn import feature_selection, impute, pipeline, preprocessing


data_preparation_pipeline = pipeline.make_pipeline(
    impute.SimpleImputer(strategy="most_frequent"),
    feature_selection.VarianceThreshold(threshold=0.0),
    preprocessing.StandardScaler(with_mean=False),
).set_output(transform="pandas")


In [ ]:
data_preparation_pipeline

## Exercise 4: Train different classifiers

Train the following classifiers. 

1. [ensemble.ExtraTreesClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.ExtraTreesClassifier.html)
2. [ensemble.HistGradientBoostingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingClassifier.html)
3. [neighbors.KNeighborsClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html)
4. [linear_model.SGDClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.SGDClassifier.html)

### Solution

In [ ]:
import joblib
import pathlib
from sklearn import decomposition, ensemble, linear_model, neighbors


NAMED_CLASSIFIERS = {
    "extratreesclassifier": pipeline.make_pipeline(
        data_preparation_pipeline,
        ensemble.ExtraTreesClassifier(
            bootstrap=True,
            criterion="log_loss",
            max_features="sqrt",
            max_samples=0.8,
            n_estimators=100,
            n_jobs=-1,
            random_state=PRNG,
        )
    ),
    "histgradientboostingclassifier": pipeline.make_pipeline(
        data_preparation_pipeline,
        ensemble.HistGradientBoostingClassifier(
            early_stopping=True,
            loss="log_loss",
            n_iter_no_change=10,
            random_state=PRNG,
            validation_fraction=0.1,
        )
    ),
    "kneighborsclassifier": pipeline.make_pipeline(
        data_preparation_pipeline,
        decomposition.PCA(
            n_components=0.95,
            whiten=True,
        ),
        neighbors.KNeighborsClassifier(
            n_jobs=-1,
            n_neighbors=3,
            weights="distance",
        )
    ),
    "sgdclassifier": pipeline.make_pipeline(
        data_preparation_pipeline,
        linear_model.SGDClassifier(
            loss="hinge",
            n_jobs=-1,
            random_state=PRNG,
        )
    ),
}

In [ ]:
for (name, classifier) in NAMED_CLASSIFIERS.items():

    model_filepath = pathlib.Path(f"/kaggle/working/{name}.pkl")

    if not model_filepath.exists():

        _ = classifier.fit(
            train_features_df,
            train_label
        )

        _ = joblib.dump(
            classifier,
            model_filepath
        )
        print(f"Done training {name}!")

    NAMED_CLASSIFIERS[name] = joblib.load(model_filepath)
    

## Exercise 5: Assess classifier performance

Use the validation set you created to assess the accuracy of the individual classifiers.

### Solution

In [ ]:
from sklearn import metrics


for (name, classifier) in NAMED_CLASSIFIERS.items():
    _val_predictions = classifier.predict(val_features_df)
    _report = metrics.classification_report(
        val_label,
        _val_predictions
    )

    print(f"{name}:")
    print(_report)

## Exercise 6: Hard voting ensemble

Combine the classifiers above into a [ensemble.VotingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.VotingClassifier.html) using hard voting. 

Does the hard-voting ensemble outperform the best individual classifier? If so, how much better does it perform compared to the best individual classifier?

### Solution

In [ ]:
_checkpoint = "/kaggle/working/hard-voting-classifier.pkl"
model_filepath = pathlib.Path(_checkpoint)

if not model_filepath.exists():
    
    _estimators = list(NAMED_CLASSIFIERS.items())

    _classifier = ensemble.VotingClassifier(
        estimators=_estimators,
        voting="hard",
        n_jobs=-1,
    )

    _ = _classifier.fit(
        train_features_df,
        train_label
    )
    
    _ = joblib.dump(
        _classifier,
        model_filepath
    )

hard_voting_classifier = joblib.load(model_filepath)

In [ ]:
_val_predictions = hard_voting_classifier.predict(val_features_df)
_report = metrics.classification_report(
    val_label,
    _val_predictions
)

print(_report)

## Exercise 7: Soft voting ensemble

Combine the classifiers above into a [ensemble.VotingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.VotingClassifier.html) using soft voting.

Remember that you will need to use [calibration.CalibratedClassifierCV](https://scikit-learn.org/stable/modules/generated/sklearn.calibration.CalibratedClassifierCV.html) to wrap [linear_model.SGDClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.SGDClassifier) in order for the `predict_proba` method required by soft-voting ensembles to be available. 

Does the soft-voting ensemble outperform the best individual classifier? If so, how much better does it perform compared to the best individual classifier?

### Solution

In [ ]:
from sklearn import calibration


NAMED_CLASSIFIERS["sgdclassifier"] = calibration.CalibratedClassifierCV(
    pipeline.make_pipeline(
        data_preparation_pipeline,
        linear_model.SGDClassifier(
            loss="hinge",
            n_jobs=-1,
            random_state=PRNG,
        ),
    ),
    cv=5,
    n_jobs=-1
)

_checkpoint = "/kaggle/working/soft-voting-classifier.pkl"
model_filepath = pathlib.Path(_checkpoint)

if not model_filepath.exists():
    
    _estimators = list(NAMED_CLASSIFIERS.items())

    _classifier = ensemble.VotingClassifier(
        estimators=_estimators,
        voting="soft",
        n_jobs=-1,
    )

    _ = _classifier.fit(
        train_features_df,
        train_label
    )
    
    _ = joblib.dump(
        _classifier,
        model_filepath
    )

soft_voting_classifier = joblib.load(model_filepath)

In [ ]:
_val_predictions = soft_voting_classifier.predict(val_features_df)
_report = metrics.classification_report(
    val_label,
    _val_predictions
)

print(_report)

## Exercise 8: Fine-tune your best classifier

Use [model_selection.GridSearchCV](https://scikit-learn.org/dev/modules/generated/sklearn.model_selection.GridSearchCV.html) to tune the hyperparameters of your best classifier on the combined training and validation sets.

### Solution

In [ ]:
# here we tune parameters of the boosting process and the individual trees
param_grid = [
    {
        "histgradientboostingclassifier__learning_rate": [1e-2, 1e-1, 1],
        "histgradientboostingclassifier__l2_regularization": [0, 1e-3, 1e-2],
        "histgradientboostingclassifier__max_depth": [None],
        "histgradientboostingclassifier__max_leaf_nodes": [31, None],
        "histgradientboostingclassifier__min_samples_leaf": [1, 20],
    }
]

# use a single train-val split
_cv = model_selection.StratifiedShuffleSplit(
    n_splits=1,
    random_state=PRNG,
    test_size=0.2
)

# use the full training set for fine-tuning
_train_features_df = TRAIN_DF.drop("label", axis=1)
_train_label = TRAIN_DF.loc[:, "label"]
_cv_splits = _cv.split(_train_features_df, _train_label)

grid_search_cv = model_selection.GridSearchCV(
    NAMED_CLASSIFIERS["histgradientboostingclassifier"],
    param_grid,
    cv=_cv_splits,
    n_jobs=-1,
    scoring="accuracy",
    verbose=2
)

_ = grid_search_cv.fit(_train_features_df, _train_label)


In [ ]:
_train_features_df = TRAIN_DF.drop("label", axis=1)
_train_label = TRAIN_DF.loc[:, "label"]



In [ ]:
grid_search_cv.best_score_

In [ ]:
grid_search_cv.best_params_

In [ ]:
best_model_filepath = pathlib.Path("/kaggle/working/best-classifier.pkl")

_ = joblib.dump(
    grid_search_cv.best_estimator_,
    best_model_filepath
)


## Exercise 9: Submit predictions

Load the sample submission file and replace the `Label` column with your best classifier predictions on the test set.

### Solution

In [ ]:
best_classifier = joblib.load(best_model_filepath)
_test_predictions = best_classifier.predict(TEST_FEATURES_DF)

_ = (
    pd.read_csv(
        "/kaggle/input/digit-recognizer/sample_submission.csv",
        index_col="ImageId"
    ).assign(
        Label=_test_predictions
    ).to_csv("submission.csv", index=True)
)

In [ ]:
%%bash

cat submission.csv | head -n 5